# 데이터 읽기

In [ ]:
from google.colab import drive
# Public portfolio note: original raw data was mounted from a private drive during the competition.

In [ ]:
# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

#warning
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#파일 읽기
pd_sas21 = pd.read_sas('../archive/raw-data/hn21_all.sas7bdat', format='sas7bdat')
pd_sas19 = pd.read_sas('../archive/raw-data/hn19_all.sas7bdat', format='sas7bdat')
pd_sas20 = pd.read_sas('../archive/raw-data/hn20_all.sas7bdat', format='sas7bdat')

# D로 시작하고 dg로 끝나는 열 중 값이 1인 행 선택(불균형데이터 방지)
subset20 = pd_sas20[(pd_sas20.filter(regex='^D.*dg$', axis=1) == 1).any(axis=1)]
subset19 = pd_sas19[(pd_sas19.filter(regex='^D.*dg$', axis=1) == 1).any(axis=1)]
subset = pd.concat([subset20, subset19])

#subset.filter(regex='^D.*dg$', axis=1)
# subset을 pd_sas19 데이터프레임에 추가
pd_sas = pd.concat([pd_sas19, subset])
pd_sas

# 전처리

## 필요없는 변수 제거

In [ ]:
#가중치 변수 제거
pd_sas = pd_sas.loc[:, ~pd_sas.columns.str.startswith('wt')]
pd_sas
#쓰레기 변수 - float, int형식 아닌 변수 제거
g_var = [col for col in pd_sas.columns if pd_sas[col].dtype not in ['float64', 'int64']]
pd_sas = pd_sas.drop(g_var, axis=1)
pd_sas
#year(조사연도) 제거
pd_sas.drop('year', axis=1, inplace=True)

In [ ]:
#값 10000개 이하로 존재하는 변수 제거
pd_sas.dropna(axis=1, thresh=10000, inplace=True)
pd_sas

In [ ]:
#결측치 3000개 이상인 변수만 출력
pd_sas.loc[:,pd_sas.isnull().sum()>3000].isnull().sum()

In [ ]:
#개인 경험과 관련된 변수 제거-LF시작 변수
pd_sas = pd_sas.loc[:, ~pd_sas.columns.str.startswith('LF')]

#영양소 섭취량 변수 중 영양소와 관련없는 변수 제거
cols_to_drop = ['N_DIET', 'N_DIET_WHY', 'N_WAT_C', 'N_PRG', 'N_BFD_Y']
for col in cols_to_drop:
    if col in pd_sas.columns:
        pd_sas.drop(col, axis=1, inplace=True)

#직업재분류 코드 제거
if 'occp' in pd_sas.columns:
  pd_sas = pd_sas.drop('occp', axis=1)

#안전벨트 착용률 제거
pd_sas = pd_sas.loc[:, ~pd_sas.columns.str.startswith('sc')]

#영양표시 제거
pd_sas = pd_sas.loc[:, ~pd_sas.columns.str.startswith('LK_')]

#식사시 동반여부*대상 변수 조사 1인전 결식여부 제거
cols_to_drop = []
for col in pd_sas.columns:
    # L_ + WHO or TO로 끝남 + 조사 1일전 식사 결식여부 변수
    if col.startswith('L_') and (col.endswith('WHO') or col.endswith('TO')):
        cols_to_drop.append(col)
if 'L_BR' in pd_sas.columns:
    cols_to_drop.append('L_BR')
if 'L_LN' in pd_sas.columns:
    cols_to_drop.append('L_LN')
if 'L_DN' in pd_sas.columns:
    cols_to_drop.append('L_DN')
pd_sas = pd_sas.drop(cols_to_drop, axis=1)

#과거력, 가족력 변수 제거 <- 생활습관과 관련되지 않은 변수여서 제거함(추후 필요시 추가 가능)
pd_sas = pd_sas.loc[:,~pd_sas.columns.str.startswith('D') | ~pd_sas.columns.str.endswith(('ag', 'pr', 'pt'))]
pd_sas = pd_sas.loc[:,~pd_sas.columns.str.startswith('HE')]

#약 변수 제거
drop_cols = [col for col in ['DI1_2', 'DI2_2', 'DJ4_3', 'LW_oc', 'HE_HPdr', 'HE_DMdr'] if col in pd_sas.columns]
if drop_cols:
    pd_sas.drop(drop_cols, axis=1, inplace=True)

#뇌졸증 후유증 제거
pd_sas.drop('DI3_2', axis=1, inplace=True)

## 결측치 처리

In [ ]:
from scipy import stats
#영양소 섭취량 NAN값은 평균치로 대체
cols_N = [col for col in pd_sas.columns if col.startswith('N_')]
for col in cols_N:
    pd_sas[col].fillna(pd_sas[col].mean(), inplace=True)

# 식품 평균 섭취량('DA_'), 빈도('DQ_')
#값이 88(비해당), 99(모름)이면 NAN으로 변환후 NAN값을 최빈값으로 대체
da_dq_cols = [col for col in pd_sas.columns if col.startswith('DA') or col.startswith('DQ')]
for col in da_dq_cols:
    pd_sas[col] = pd_sas[col].replace([88, 99], np.nan)
    mode_val = pd_sas[col].mode()[0]
    pd_sas[col].fillna(mode_val, inplace=True)

#버섯류, 채소류 등 섭취빈도('LS_')
# 99(모름)를 NAN으로 변환 후 NAN값을 최빈값으로 대체
ls_cols = [col for col in pd_sas.columns if col.startswith('LS')]
pd_sas.loc[:, ls_cols] = pd_sas.loc[:, ls_cols].replace(99, np.nan)
for col in ls_cols:
    mode_value = stats.mode(pd_sas[col])[0][0]
    pd_sas[col] = pd_sas[col].fillna(mode_value)

#식사 빈도('L_xx_FQ')
#9(모름) NAN변환 후 NAN값을 최빈값으로 대체
cols_to_check = [col for col in pd_sas.columns if col.startswith('L_') and col.endswith('FQ')]
for col in cols_to_check:
    pd_sas[col].replace({9: np.nan}, inplace=True)
    mode_value = pd_sas[col].mode()[0]
    pd_sas[col].fillna(mode_value, inplace=True)


In [ ]:
#출력값 수정
# 값이 8(비해당)이나 9(모름)인 경우 0으로 변환합니다.
cols_to_modify = [col for col in pd_sas.columns if col.startswith('D') and col.endswith('dg')]
pd_sas[cols_to_modify] = pd_sas[cols_to_modify].replace([8, 9], 0)

## 결측치 제거

In [ ]:
#결측치 3000개 이상인 변수 제거
cols_to_drop=[]
for col in pd_sas.columns:
    if pd_sas[col].isna().sum() >= 3000:
        cols_to_drop.append(col)
pd_sas.drop(cols_to_drop, axis=1, inplace=True)
pd_sas

In [ ]:
#NAN존재하는 행 제거
pd_sas.dropna(inplace=True)
pd_sas

In [ ]:
# D로 시작하고 dg로 끝나는 열을 선택하여 subset 변수에 저장
subset = pd_sas.filter(regex='^D.*dg$', axis=1)

# 값이 1인 개수를 변수별로 출력
print(subset.eq(1).sum())

# 의미있는 변수 찾기

## VIF(다중공선성) 제거

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

y = pd_sas.filter(regex="^D.*dg$")
X = pd_sas.drop(columns=y.columns)

# VIF 값이 10 이상인 feature를 모두 제거하고 VIF 값을 다시 계산
while True:
    # VIF 값과 각 Feature 이름에 대해 설정
    vif = pd.DataFrame()
    vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    vif["features"] = X.columns

    # VIF 값이 10 이상인 feature를 모두 찾음
    high_vif_features = vif.loc[vif["VIF Factor"] > 10, "features"]

    # VIF 값이 10 이상인 feature가 없는 경우, 루프 종료
    if len(high_vif_features) == 0:
        break

    # VIF 값이 10 이상인 feature를 데이터프레임에서 제거
    X = X.drop(columns=high_vif_features)

# 최종적으로 남은 feature 출력
print(X.columns, len(X.columns))

# 예측모델 생성

## XGboost

In [ ]:
import xgboost as xgb
from xgboost import plot_importance
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
import warnings
warnings.filterwarnings('ignore')

In [ ]:
params = {'max_depth': 6,#트리 깊이
          'eta': 0.1,
          'subsample': 0.8,#너무 작으면 underfitting, 너무 크면 과적합 -> 0.5 ~ 0.8
          'colsample_bytree': 0.8,#0.5~0.9
          'objective': 'binary:logistic',
          'eval_metric': 'logloss',
          'min_child_weight': 2 #너무 작으면 과적합, 너무 크면 underfitting 1~20
}
num_rounds = 1000#학습진행할때 몇번 반복할것인가 너무 크면 과적합 -> 조금씩 수정

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score, roc_auc_score
import copy

In [ ]:
ls=[]
disease_features={}#질병-독립변수 리스트 쌍 저장
models={} # y변수 별로 예측모델 저장
fitted_ycols=[] #f1_score 0.5 이상인 예측모델의 key리스트
for i in y.columns:
    dataset = copy.deepcopy(pd_sas)
    X_features = dataset[X.columns]
    y_label = dataset.loc[:,i]
    
    # 변수 선택을 위한 xgb 모델 생성 및 학습
    select_xgb_model = xgb.XGBClassifier(**params, n_jobs=-1)
    select_xgb_model.fit(X_features, y_label)
    
    # SelectFromModel을 사용하여 변수 선택
    selection = SelectFromModel(select_xgb_model, threshold=-np.inf, max_features=20, prefit=False)
    selection.fit(X_features, y_label)

    # 선택된 변수만 추출하여 학습 데이터와 테스트 데이터 생성
    selected_features = X.columns[selection.get_support()]
    disease_features[i] = selected_features.tolist()# 선택된 독립변수를 disease_features 딕셔너리에 저장
    X_features_selected = selection.transform(X_features)
    X_train,X_test,y_train,y_test = train_test_split(X_features_selected, y_label, test_size=0.3, random_state=156)
    dtrain = xgb.DMatrix(data=X_train, label=y_train)
    dtest = xgb.DMatrix(data=X_test, label=y_test)

    # train 데이터 세트는 'train', evaluation(test) 데이터 세트는 'eval'로 명기
    wlist = [(dtrain,'train'),(dtest,'eval')]
    # 하이퍼 파라미터와 early stopping 파라미터를 train() 함수의 파라미터로 전달
    xgb_model = xgb.train(params=params, dtrain=dtrain, num_boost_round=num_rounds, early_stopping_rounds=500, evals=wlist)
    
    pred_probs = xgb_model.predict(dtest)
    preds = [1 if x > 0.5 else 0 for x in pred_probs]


    def get_clf_eval(y_test,pred=None,pred_proba=None):
        confusion = confusion_matrix(y_test,pred).tolist()
        accuracy = accuracy_score(y_test,pred)
        precision = precision_score(y_test,pred)
        recall = recall_score(y_test,pred)
        F1 = f1_score(y_test, pred)
        a = [confusion,accuracy,precision,recall,F1]
        
        #F1_score 0.5 이상인 예측모델
        if F1 >= 0.5:
          ls.append(a)
          models[i]=copy.deepcopy(xgb_model)
          fitted_ycols.append(i)
    get_clf_eval(y_test,preds,pred_probs)
    fig, ax = plt.subplots(figsize=(10,12))
    plot_importance(xgb_model, ax=ax)


In [ ]:
final_df = pd.DataFrame(ls,index=fitted_ycols, columns=['오차 행렬','정확도','예측률','재현율','F1'])
final_df

## 변수 선택 결과

In [ ]:
#데이터 학습에 사용된 변수 모아보기
disease_keys = disease_features.keys()
union_ls = []
for key in disease_keys:
    union_ls.append(disease_features[key])

union_ls

In [ ]:
import itertools

# 이중 리스트를 하나의 리스트로 합치기
merged_list = list(itertools.chain(*union_ls))

# 중복된 값을 제거한 후 리스트로 변환하기
union_list = list(set(merged_list))

# 결과 출력하기
union_list, len(union_list)

# Input 데이터로 보험 추천

## INPUT 데이터 생성후 질병 예측

In [ ]:
#예시 INPUT 데이터
input_data = X[union_list].apply(np.random.choice, axis=0)
input_data

In [ ]:
predicted_ps = pd.DataFrame()
find_diceases=0 #질병예측

while find_diceases == 0:
  predicted_ps = pd.DataFrame()
  input_data = X[union_list].apply(np.random.choice, axis=0)
  for code in final_df.index:
    selected_features = disease_features[code]
    dInput=xgb.DMatrix(data=input_data[selected_features].values.reshape(1,-1))
    predicted_ps[code]=models[code].predict(dInput)
  predicted_ps = predicted_ps.T
  find_diceases = (predicted_ps >= 0.5).sum().sum() #예측결과 확률이 0.5이상인 질병의 개수
predicted_ps = predicted_ps.rename(columns={0:'확률'})
predicted_ps

## 보험추천

In [ ]:
insure_1 = 0
insure_2 = 0
insure_3 = 0

# 기존 데이터프레임(병코드에 맞는 병명을 추가하기 위함입니다.)
original_df = pd.read_excel('../archive/raw-data/original_df.xlsx',index_col=0)

# 두 데이터프레임을 변수를 기준으로 합침
merged_df = pd.concat([original_df, predicted_ps['확률']],axis=1).dropna()

# 합친 데이터프레임에서 군집 값을 가져와서 중복되는 값을 제거하여 "군집" 칼럼을 업데이트 함
# result_df = merged_df[["레이블", "확률", "군집"]]

for idx, row in merged_df.iterrows():
    if row["군집"] == 1:
        insure_1 += row["확률"]
    elif row["군집"] == 2:
        insure_2 += row["확률"]

insure_3 = (insure_1+insure_2)/2

#insure_1 : 뇌심혈관 질병의 확률을 더합니다.
#insure_2 : 암 질병의 확률을 더합니다.
#insure_3 : insure_1과 insure2의 평균을 냅니다.

#뇌심혈과 질병과 암 질병 외의 질병은 모두 "기타" 로 분류했습니다.
#신한라이프 홈페이지에 보험설명이 나와있는 보험상품중 상해보험과 종신보험을 제외한 보험을 추천하기 위함입니다.

print(f"insure_1: {insure_1}")
print(f"insure_2: {insure_2}")
print(f"insure_3: {insure_3}")

In [ ]:
mreged_df = merged_df.sort_values(by="확률", ascending=False)
#"확률"칼럼을 기준으로 내림차순합니다.

data1, data2, data3, data4, data5 = merged_df.iloc[:5]["레이블"]
p1, p2, p3, p4, p5 = merged_df.iloc[:5]["확률"]
#상위 5개까지 끊습니다.

max_value = max(insure_1, insure_2, insure_3)
insure_name = " "
if max_value == insure_1:
  insure_name = "신한 평생간병비 걱정없는 뇌심혈관보험(무배당, 해약환급금 일부지급형)"
elif max_value == insure_2:
  insure_name = "신한 3COLOR 암플러스 보장보험(무배당, 해약환급금 일부지급형)"
elif max_value == insure_3:
  insure_name = "신한 3COLOR 3대질병 보장보험(무배당, 갱신형)"

#고객 특성과 보험상품의 특성에 맞춰 보험을 추천합니다.
#inusre_1, insure_2, insure_3중 가장 큰 값을 찾아서 그 값이 특정 임계를 넘으면 고객특성이라고 판단하고 추천합니다.
#특정 임계를 넘지 못했을경우, 가장 확률이 높은 질병 관련 보험을 추천합니다.
#예상질병과 확률 상위5개를 출력합니다.

print("고객님의 예상질병과 확률은 다음과 같습니다.")
print(data1, round(p1, 4)*100,"%")
print(data2, round(p2, 4)*100,"%")
print(data3, round(p3, 4)*100,"%")
print(data4, round(p4, 4)*100,"%")
print(data5, round(p5, 4)*100,"%")
print("\n")

if(max_value>0.5):
  print('저희가 추천하는 보험은 "', insure_name,'"')
else:
  print("저희가 추천하는 보험은", data1, "관련 보험입니다.")